# 数字人口播工作台 · Colab 直跑版(无需 frp)

**全部运行**(代码执行程序 → 全部运行)即可:装好 MuseTalk + CosyVoice 后,最后一个 cell 会直接打印一个 **公网 https 地址**(cloudflared),手机/电脑打开就是完整工作台。

可选 Secrets(左侧 🔑,开 Notebook access):
- `GROQ_API_KEY`:文案提取转写 / SRT 字幕 / 音色参考转写用
- `OPENAI_API_KEY` + `OPENAI_BASE_URL` + `OPENAI_MODEL`:AI 仿写 / 标题用

不配也能跑通「配音 → 对口型 → 剪辑」主链路。

In [ ]:
#@title 1️⃣ 安装 MuseTalk + CosyVoice(后台并行,约 20-40 分钟)
import subprocess
cmds = [
 "mkdir -p /content/mt; curl -fsSL https://raw.githubusercontent.com/cicy-ai/cicy-tools/main/musetalk-provision.sh -o /content/mt/provision.sh; nohup bash /content/mt/provision.sh > /content/mt/provision.log 2>&1 & echo musetalk provisioning...",
 "mkdir -p /content/cosy; curl -fsSL https://raw.githubusercontent.com/cicy-ai/cicy-tools/main/cosyvoice-provision.sh -o /content/cosy/provision.sh; nohup bash /content/cosy/provision.sh > /content/cosy/provision.log 2>&1 & echo cosyvoice provisioning...",
 "which node || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1 && apt-get install -y -q nodejs >/dev/null)",
 "node -v && npm -v",
]
for c in cmds:
    subprocess.run(c, shell=True, executable="/bin/bash")
print("✅ 两套环境在后台安装;可先运行下一个 cell 起工作台(环境好之前先用不吃 GPU 的功能)。")

In [ ]:
#@title 2️⃣ 启动工作台 + 公网地址(保持此 cell 运行)
# 从 Secrets 注入可选密钥(没配就跳过)
import os
from google.colab import userdata
for k in ("GROQ_API_KEY", "OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_MODEL"):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception:
        pass
# 本 cell 会一直运行(即出片服务);输出里的 trycloudflare 地址就是入口(生效约需 1 分钟)
!KOUBO_NO_OPEN=1 npx -y cicy-koubo --cft

In [ ]:
#@title 📈 安装进度(可反复运行)
import subprocess
sh = lambda c: print(subprocess.run(c, shell=True, executable="/bin/bash",
                     capture_output=True, text=True).stdout, end="")
sh('echo "== MuseTalk:"; grep -E "^===|^!!!" /content/mt/provision.log 2>/dev/null | tail -3')
sh('test -f /content/mt/READY && echo "   ✅ MuseTalk READY" || echo "   ⏳ 安装中…"')
sh('echo "== CosyVoice:"; grep -E "^===|^!!!" /content/cosy/provision.log 2>/dev/null | tail -3')
sh('test -f /content/cosy/COSY_READY && echo "   ✅ CosyVoice READY" || echo "   ⏳ 安装中…"')

In [ ]:
#@title (可选)安装 HeyGem 引擎 — 实验,需 ≥16GB 显存(A100/L4;T4 不够)
import subprocess
subprocess.run("mkdir -p /content/hg; curl -fsSL https://raw.githubusercontent.com/cicy-ai/cicy-tools/main/heygem-provision.sh -o /content/hg/provision.sh; "
               "nohup bash /content/hg/provision.sh > /content/hg/provision.log 2>&1 & echo heygem provisioning...",
               shell=True, executable="/bin/bash")
print("HeyGem 后台安装中(权重较大,约 20-40 分钟);进度看 /content/hg/provision.log,完成写 /content/hg/HG_READY。")
print("装好后在工作台「对口型引擎」里选 HeyGem 即可。")